# Example 2.4.1 / 1: Labeling a NumPy Matrix

Companion notebook for **2.4.1 Data from NumPy and Polars** in the AMPL lab Documentation, section "Labeling a NumPy Matrix". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Labeling a NumPy Matrix

A dense array supplies values but not the meaning of its axes. Build a DataFrame with row and
column labels before assigning a two-dimensional parameter.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;
set DEST;
param cost {ORIG, DEST} >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
import numpy as np
import pandas as pd

origins = ["GARY", "CLEV"]
destinations = ["FRA", "DET"]
matrix = np.array([[39, 14], [27, 9]])

cost = pd.DataFrame(matrix, index=origins, columns=destinations)
ampl.set["ORIG"] = origins
ampl.set["DEST"] = destinations
ampl.param["cost"] = cost

In [5]:
# run.py
print(ampl.param["cost"]["CLEV", "DET"])

9


### Pure AMPL: data.dat and model.run

Assigning the wide DataFrame to `cost` reads its index as `ORIG` and its columns as `DEST`.

The equivalent AMPL table carries the same labels:

In [6]:
%%writefile data.dat
set ORIG := GARY CLEV;
set DEST := FRA DET;
param cost: FRA DET :=
  GARY 39 14
  CLEV 27  9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display cost['CLEV','DET'];

cost['CLEV','DET'] = 9

